In [1]:
import snntorch as snn
import torch
import torch.nn as nn
import os
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import random
import torch.optim as optim
import dotenv   
print("Done")

Done


In [2]:
dotenv.load_dotenv()
op="output"

In [3]:
def pt_loader(path):
    return torch.load(path)

# DatasetFolder scans for your newly saved files automatically
train_dataset = datasets.DatasetFolder(
    root=os.path.join(op, "Train"),
    loader=pt_loader,
    extensions = ('.pt',)
)

test_dataset = datasets.DatasetFolder(
    root=os.path.join(op, "Test"),
    loader=pt_loader,
    extensions = ('.pt',)
)
val_dataset = datasets.DatasetFolder(
    root=os.path.join(op, "Validate"),
    loader=pt_loader,
    extensions=('.pt',)
)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
print(f"Number of samples in training : {len(train_dataset)}")
print(f"Number of samples in test : {len(test_dataset)}")
print(f"Number of samples in validation : {len(val_dataset)}")

Number of samples in training : 30003
Number of samples in test : 4999
Number of samples in validation : 5001


In [17]:
class SpikingDigitCNN(nn.Module):
    def __init__(self):
        super().__init__()

        beta = 0.8  # Decay
        spike_grad = snn.surrogate.fast_sigmoid(slope=25)

        # Layer 1
        self.conv1 = nn.Conv2d(in_channels=2, out_channels=16, kernel_size=3, padding=1)
        self.mp1 = nn.MaxPool2d(kernel_size=2)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad,threshold=0.5)

        # Layer 2
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.mp2 = nn.MaxPool2d(kernel_size=2)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad,threshold=0.5)

        # Classification
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad,threshold=0.5)

        self.fc2 = nn.Linear(128, 10)
        self.lif4 = snn.Leaky(beta=beta, spike_grad=spike_grad,threshold=0.5)

    def forward(self, x):
        # Expected (from your .pt files): [Batch, Channels=2, Time=30, H, W]
        if x.dim() == 5:
            x = x.permute(2, 0, 1, 3, 4)
        

        num_steps = x.shape[0]

        # Reset cell potentials
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()

        spk4_recording = []

        for step in range(num_steps):
            current_frame = x[step]  # [Batch, Channels, H, W]

            cur1 = self.mp1(self.conv1(current_frame))
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.mp2(self.conv2(spk1))
            spk2, mem2 = self.lif2(cur2, mem2)

            cur3 = self.fc1(self.flatten(spk2))
            spk3, mem3 = self.lif3(cur3, mem3)

            cur4 = self.fc2(spk3)
            spk4, mem4 = self.lif4(cur4, mem4)

            spk4_recording.append(spk4)

        return torch.stack(spk4_recording)  # [Time, Batch, Classes]

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        print(inputs.shape, targets.shape)
        break

torch.Size([64, 2, 30, 34, 34]) torch.Size([64])


In [18]:
import snntorch.functional as SF


model = SpikingDigitCNN().to(device)


loss_func = SF.mse_count_loss(correct_rate=0.8, incorrect_rate=0.2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

epochs = 10



In [19]:
train_loss_history = np.array([])
test_loss_history = np.array([])

for epoch in range(epochs):

    model.train()
    running_train_loss = 0.0
    train_total = 0  
    
    for X_train, y_train in train_loader:
        X_train, y_train = X_train.to(device), y_train.long().to(device)
        
        y_pred = model(X_train)
        loss = loss_func(y_pred, y_train)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * X_train.size(0)
        train_total += y_train.size(0)  
        
    avg_train_loss = running_train_loss / train_total
    train_loss_history = np.append(train_loss_history, avg_train_loss)
   

    model.eval()
    running_test_loss = 0.0
    test_total = 0  
    
    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(device), y_test.long().to(device)
            
            y_test_pred = model(x_test)
            loss = loss_func(y_test_pred, y_test)
            
            running_test_loss += loss.item() * x_test.size(0)
            test_total += y_test.size(0)  
            
    avg_test_loss = running_test_loss / test_total
    test_loss_history = np.append(test_loss_history, avg_test_loss)
       
    print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")



Epoch [01/10] | Train Loss: 0.6823 | Test Loss: 0.4057
Epoch [02/10] | Train Loss: 0.3477 | Test Loss: 0.3079
Epoch [03/10] | Train Loss: 0.2781 | Test Loss: 0.2597
Epoch [04/10] | Train Loss: 0.2407 | Test Loss: 0.2314
Epoch [05/10] | Train Loss: 0.2134 | Test Loss: 0.2118
Epoch [06/10] | Train Loss: 0.1917 | Test Loss: 0.1798
Epoch [07/10] | Train Loss: 0.1752 | Test Loss: 0.1705
Epoch [08/10] | Train Loss: 0.1626 | Test Loss: 0.1569
Epoch [09/10] | Train Loss: 0.1509 | Test Loss: 0.1508
Epoch [10/10] | Train Loss: 0.1413 | Test Loss: 0.1373


In [20]:
import numpy as np
import torch

# Put model into evaluation mode (turns off dropout, freeze batchnorm)
model.eval()

# ==================== 1. PROCESS TRAINING DATA ====================
train_predicted_list = []
train_actual_list = []
cnt = 0

print("--- EVALUATING TRAINING LOADER ---")
with torch.no_grad():
    for X_train, y_train in train_loader:
        X_train = X_train.to(device)
        
        # SNN Forward Pass outputs shape: [Time=30, Batch, Classes=10]
        y_train_pred_raw = model(X_train)
        
        # Collapse time dimension by summing spikes over the 30 steps
        train_summed_spikes = y_train_pred_raw.sum(dim=0)
        
        # Extract the predicted class index
        y_train_pred = torch.argmax(train_summed_spikes, dim=1).cpu().numpy()
        
        train_predicted_list.extend(y_train_pred)
        train_actual_list.extend(y_train.numpy())
        
        if cnt % 20 == 0:
            print(f"Processing Train Batch {cnt}...")
        cnt += 1

train_predicted = np.array(train_predicted_list)
train_actual = np.array(train_actual_list)


# ==================== 2. PROCESS VALIDATION DATA ====================
val_predicted_list = []
val_actual_list = []
cnt = 0

print("\n--- EVALUATING VALIDATION LOADER ---")
with torch.no_grad():
    for X_val, y_val in val_loader:
        X_val = X_val.to(device)
        
        # SNN Forward Pass
        y_val_pred_raw = model(X_val)
        
        # Collapse time dimension by summing spikes over the 30 steps
        val_summed_spikes = y_val_pred_raw.sum(dim=0)
        
        # Extract the predicted class index
        y_val_pred = torch.argmax(val_summed_spikes, dim=1).cpu().numpy()
        
        val_predicted_list.extend(y_val_pred)
        val_actual_list.extend(y_val.numpy())
        
        if cnt % 20 == 0:
            print(f"Processing Val Batch {cnt}...")
        cnt += 1

val_predicted = np.array(val_predicted_list)
val_actual = np.array(val_actual_list)

print("\n--- MATRIX EVALUATION ARRAYS GENERATED SUCCESSFULLY ---")

--- EVALUATING TRAINING LOADER ---
Processing Train Batch 0...
Processing Train Batch 20...
Processing Train Batch 40...
Processing Train Batch 60...
Processing Train Batch 80...
Processing Train Batch 100...
Processing Train Batch 120...
Processing Train Batch 140...
Processing Train Batch 160...
Processing Train Batch 180...
Processing Train Batch 200...
Processing Train Batch 220...
Processing Train Batch 240...
Processing Train Batch 260...
Processing Train Batch 280...
Processing Train Batch 300...
Processing Train Batch 320...
Processing Train Batch 340...
Processing Train Batch 360...
Processing Train Batch 380...
Processing Train Batch 400...
Processing Train Batch 420...
Processing Train Batch 440...
Processing Train Batch 460...

--- EVALUATING VALIDATION LOADER ---
Processing Val Batch 0...
Processing Val Batch 20...
Processing Val Batch 40...
Processing Val Batch 60...

--- MATRIX EVALUATION ARRAYS GENERATED SUCCESSFULLY ---


In [21]:
from sklearn.metrics import classification_report, confusion_matrix

# Print Training performance profile
print("================ TRAINING REPORT ================")
print(classification_report(train_actual, train_predicted))

# Print Validation performance profile
print("================ VALIDATION REPORT ================")
print(classification_report(val_actual, val_predicted))

print("================ VALIDATION CONFUSION MATRIX ================")
print(confusion_matrix(val_actual, val_predicted))

================ TRAINING REPORT ================
              precision    recall  f1-score   support

           0       0.96      0.97      0.97      2962
           1       0.97      0.98      0.97      3371
           2       0.95      0.95      0.95      2979
           3       0.93      0.93      0.93      3066
           4       0.95      0.96      0.95      2921
           5       0.95      0.94      0.94      2711
           6       0.97      0.97      0.97      2959
           7       0.93      0.96      0.95      3133
           8       0.93      0.91      0.92      2926
           9       0.95      0.90      0.92      2975

    accuracy                           0.95     30003
   macro avg       0.95      0.95      0.95     30003
weighted avg       0.95      0.95      0.95     30003

================ VALIDATION REPORT ================
              precision    recall  f1-score   support

           0       0.94      0.99      0.96       490
           1       0.98      0

In [23]:
import time
import torch
import numpy as np
from thop import profile


def profile_snn_performance(model, device):
    model.eval()

    # ---------------------------------------------------------
    # 1. MODEL SIZE PROFILE
    # ---------------------------------------------------------
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    disk_size_mb = (trainable_params * 4) / (1024 * 1024)

    print("================ SNN PARAMETER PROFILE ================")
    print(f"Total Parameters:          {total_params:,}")
    print(f"Trainable Parameters:      {trainable_params:,}")
    print(f"Estimated Size on Disk:    {disk_size_mb:.2f} MB\n")

    # ---------------------------------------------------------
    # 2. INFERENCE LATENCY & RUNTIME MEMORY
    # ---------------------------------------------------------
    # Match your dataset tensor shape: [Batch=1, Channels=2, Time=30, H=34, W=34]
    dummy_stream = torch.randn(1, 2, 30, 34, 34, device=device)

    # Warm-up pass to eliminate initial allocation lag
    with torch.inference_mode():
        _ = model(dummy_stream)

    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)

    # Measure latency over 100 iterations
    start_time = time.perf_counter()
    with torch.inference_mode():
        for _ in range(100):
            _ = model(dummy_stream)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    end_time = time.perf_counter()

    avg_latency = ((end_time - start_time) / 100) * 1000
    print("================ SNN RUNTIME INFERENCE ================")
    print(f"Inference Latency:         {avg_latency:.4f} ms per event stream")

    if device.type == 'cuda':
        peak_vram = torch.cuda.max_memory_allocated(device) / (1024 * 1024)
        print(f"Runtime Memory Footprint:  {peak_vram:.2f} MB\n")
    else:
        print("Runtime Memory Footprint:  (Switch to GPU/CUDA to measure exact VRAM footprint)\n")

    # ---------------------------------------------------------
    # 3. COMPUTATIONAL WORKLOAD & ENERGY ESTIMATION
    # ---------------------------------------------------------
    # Note: THOP will run the full forward pass (including the 30-step loop)
    macs_per_stream, _ = profile(model, inputs=(dummy_stream,), verbose=False)

    # Rough energy estimate (commonly cited ballpark for MAC on 45nm): ~4.6 pJ
    estimated_joules = macs_per_stream * 4.6 * 1e-12

    print("================ ESTIMATED COMPUTATIONAL COST ================")
    print(f"Total Synaptic Operations: {int(macs_per_stream):,} ops per inference stream")
    print(f"Estimated Energy Cost:     {estimated_joules:.8f} Joules")
    print("==============================================================")


# Execute the profiler
profile_snn_performance(model, device)

================ SNN PARAMETER PROFILE ================
Total Parameters:          268,506
Trainable Parameters:      268,506
Estimated Size on Disk:    1.02 MB

================ SNN RUNTIME INFERENCE ================
Inference Latency:         16.3930 ms per event stream
Runtime Memory Footprint:  142.74 MB

================ ESTIMATED COMPUTATIONAL COST ================
Total Synaptic Operations: 57,841,920 ops per inference stream
Estimated Energy Cost:     0.00026607 Joules
